### Jaccard 

In [ ]:
import json
from pathlib import Path
from itertools import combinations

FOLDER = r"E:\TUB\WS25-26\fiz\metadata_extraction"
json_files = list(Path(FOLDER).glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {FOLDER}")

all_data = {}  # model_name -> {field: value}
for f in json_files:
    with open(f, 'r', encoding='utf-8') as fp:
        data = json.load(fp)
    if isinstance(data, list):
        entry = data[0]
    else:
        entry = data
    model = entry.get('model', f.stem)
    meta = entry.get('result', {}).get('metadata', [])
    field_value = {item['field']: item.get('value', '') for item in meta}
    all_data[model] = field_value

model_names = list(all_data.keys())
print(f"Loaded models: {model_names}\n")

# 2. Compute pairwise Jaccard similarity between models
pairwise_jaccards = []
for m1, m2 in combinations(model_names, 2):
    set1 = set(all_data[m1].keys())
    set2 = set(all_data[m2].keys())
    inter = len(set1 & set2)
    union = len(set1 | set2)
    j = inter / union if union else 0.0
    pairwise_jaccards.append(j)
    print(f"Jaccard({m1}, {m2}) = {j:.4f}")

avg_jaccard = sum(pairwise_jaccards) / len(pairwise_jaccards) if pairwise_jaccards else 0.0
print(f"\nAverage pairwise Jaccard (inter‑model consistency) = {avg_jaccard:.4f}")

Loaded models: ['anthropic/claude-opus-4.8', 'deepseek/deepseek-v4-pro', 'qwen/qwen3.7-plus']

Jaccard(anthropic/claude-opus-4.8, deepseek/deepseek-v4-pro) = 0.1346
Jaccard(anthropic/claude-opus-4.8, qwen/qwen3.7-plus) = 0.2564
Jaccard(deepseek/deepseek-v4-pro, qwen/qwen3.7-plus) = 0.1628

Average pairwise Jaccard (inter‑model consistency) = 0.1846


### Cohen’s kappa

In [11]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

In [12]:
# 2. set of all unique fields across all models
all_fields = set().union(*[set(d.keys()) for d in all_data.values()])
all_fields = sorted(all_fields)
print(f"Total candidate fields (union): {len(all_fields)}\n")

# 3. set up binary matrix: rows = models, columns = fields, 1 if field present in model, else 0
binary_matrix = []
for model in model_names:
    field_set = set(all_data[model].keys())
    vec = [1 if f in field_set else 0 for f in all_fields]
    binary_matrix.append(vec)

# 4. compute pairwise Cohen's kappa between models
pairwise_kappa = []
for i, j in combinations(range(len(model_names)), 2):
    kappa = cohen_kappa_score(binary_matrix[i], binary_matrix[j])
    pairwise_kappa.append(kappa)
    print(f"Cohen's kappa ({model_names[i]} vs {model_names[j]}) = {kappa:.4f}")

avg_kappa = np.mean(pairwise_kappa) if pairwise_kappa else 0.0
print(f"\nAverage pairwise Cohen's kappa (≈ Fleiss' kappa) = {avg_kappa:.4f}")

# 5. Positive Agreement
def positive_agreement(vec1, vec2):
    both1 = sum((v1 == 1 and v2 == 1) for v1, v2 in zip(vec1, vec2))
    sum1 = sum(vec1)
    sum2 = sum(vec2)
    if sum1 + sum2 == 0:
        return 1.0  
    return 2 * both1 / (sum1 + sum2)

pairwise_pos = []
for i, j in combinations(range(len(model_names)), 2):
    pa = positive_agreement(binary_matrix[i], binary_matrix[j])
    pairwise_pos.append(pa)
    print(f"Positive agreement ({model_names[i]} vs {model_names[j]}) = {pa:.4f}")

avg_pos = np.mean(pairwise_pos) if pairwise_pos else 0.0
print(f"\nAverage positive agreement = {avg_pos:.4f}")

Total candidate fields (union): 61

Cohen's kappa (anthropic/claude-opus-4.8 vs deepseek/deepseek-v4-pro) = -0.4766
Cohen's kappa (anthropic/claude-opus-4.8 vs qwen/qwen3.7-plus) = 0.0328
Cohen's kappa (deepseek/deepseek-v4-pro vs qwen/qwen3.7-plus) = -0.1870

Average pairwise Cohen's kappa (≈ Fleiss' kappa) = -0.2103
Positive agreement (anthropic/claude-opus-4.8 vs deepseek/deepseek-v4-pro) = 0.2373
Positive agreement (anthropic/claude-opus-4.8 vs qwen/qwen3.7-plus) = 0.4082
Positive agreement (deepseek/deepseek-v4-pro vs qwen/qwen3.7-plus) = 0.2800

Average positive agreement = 0.3085
